# Equipo 19 · Avance 6: aplicar los modelos end-to-end a un video de C3VDv2

Este notebook **no entrena nada**. Toma los dos modelos finales del Avance 5 (Endo-STTN y AF-SfMLearner entrenados
juntos, uno por máscara) y los aplica a **cualquier video** de C3VDv2, frame por frame y en orden:

```
video ─► máscara de reflejos ─► Endo-STTN ajustado (inpainting) ─► frame limpio ─► estimador ajustado ─► profundidad
         (El Meslouhi o Arnold,      sttn.pth                                       encoder.pth + depth.pth
          según el modelo)
```

| Salida | Qué es |
|---|---|
| Visor frame por frame | Un control deslizante para recorrer el video: original, limpio y profundidad de cada modelo |
| Video `.mp4` | Todo el video lado a lado, para la presentación |
| Métricas (solo si el video trae `depth/`) | AbsRel en la imagen completa y en los reflejos, por frame y en promedio, y una medida de parpadeo |

**Para que sea una evaluación válida**, el video no debe haberse usado para entrenar ni validar (ni ser otra
versión del mismo segmento). El notebook lo revisa y avisa.

## 0. Configuración

**Colab Pro:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución →* **A100 GPU** (o L4) y **RAM alta**.
Después: *Entorno de ejecución → Ejecutar todas*.

| En Google Drive | Qué es |
|---|---|
| `MyDrive/Equipo19/raw/<video>.zip` | El video de C3VDv2 que quieres evaluar (con `rgb/`; `depth/` es opcional) |
| `MyDrive/Equipo19/end_to_end_mascaras/modelos/E_meslouhi_s0/` y `E_arnold_s0/` | Los modelos del Avance 5: `sttn.pth`, `encoder.pth`, `depth.pth` |
| `MyDrive/Equipo19/Model_trained_end_to_end.zip` | Pesos SCARED de AF-SfMLearner, solo para la referencia A. Si no está, se descarga |

Los resultados se guardan en `MyDrive/Equipo19/avance6/<video>/`.

In [ ]:
from pathlib import Path
import os, sys, json, re, shutil, subprocess, time, zipfile

# ---------- Ajusta aquí ----------
VIDEO = "c2_descending_t4_v3"     # nombre del ZIP en CARPETA_RAW, sin ".zip"
PASO = 1                          # 1 = todos los frames (video fluido); 5 = 1 de cada 5 (más rápido)
MAX_FRAMES = 0                    # 0 = sin límite; p. ej. 300 para una prueba rápida
INCLUIR_A = True                  # referencia A: estimador original (SCARED) sobre frames sin limpiar
FPS = 30                          # cuadros por segundo del .mp4 (con PASO = 5, usa 6 para conservar la duración)
BLOQUE = 60                       # frames por bloque al limpiar (bájalo si falta memoria)

MYDRIVE = Path("/content/drive/MyDrive")
CARPETA_RAW = MYDRIVE / "Equipo19" / "raw"
MODELOS = MYDRIVE / "Equipo19" / "end_to_end_mascaras" / "modelos"
MODELOS_E = {"meslouhi": MODELOS / "E_meslouhi_s0", "arnold": MODELOS / "E_arnold_s0"}   # máscara -> carpeta
SALIDA = MYDRIVE / "Equipo19" / "avance6" / VIDEO
# ---------------------------------

assert "google.colab" in sys.modules, "Este notebook es para Google Colab"
from google.colab import drive
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "tabulate"], check=True)

LOCAL = Path("/content")
EXTERNOS = LOCAL / "externos"
EXTERNOS.mkdir(parents=True, exist_ok=True)
SALIDA.mkdir(parents=True, exist_ok=True)

# Arquitecturas publicadas, fijadas a la versión que usa el equipo
REPOS = {"Endo-STTN": ("https://github.com/endomapper/Endo-STTN.git", "9419a6c"),
         "AF-SfMLearner": ("https://github.com/ShuweiShao/AF-SfMLearner.git", "53d01a4")}
for nombre, (url, commit) in REPOS.items():
    if not (EXTERNOS / nombre).exists():
        subprocess.run(["git", "clone", "-q", url, str(EXTERNOS / nombre)], check=True)
        subprocess.run(["git", "-C", str(EXTERNOS / nombre), "checkout", "-q", commit], check=True)
    if str(EXTERNOS / nombre) not in sys.path:
        sys.path.append(str(EXTERNOS / nombre))

MODELOS_E = {m: c for m, c in MODELOS_E.items()
             if all((c / f).exists() for f in ("sttn.pth", "encoder.pth", "depth.pth"))}
assert MODELOS_E, f"No encuentro modelos E completos en {MODELOS}"
print("Modelos E:", {m: str(c) for m, c in MODELOS_E.items()})

PESOS_SCARED = EXTERNOS / "pesos_scared"
if INCLUIR_A and not (PESOS_SCARED / "encoder.pth").exists():
    z = MYDRIVE / "Equipo19" / "Model_trained_end_to_end.zip"
    if not z.exists():
        import gdown
        z = EXTERNOS / "Model_trained_end_to_end.zip"
        gdown.download(id="1ZogZ5nQsfLhFAQGG_UE9A5R_osypLZf1", output=str(z), quiet=True)
    shutil.unpack_archive(str(z), str(EXTERNOS / "pesos_zip"))
    origen = next((EXTERNOS / "pesos_zip").rglob("encoder.pth")).parent
    PESOS_SCARED.mkdir(exist_ok=True)
    for f in ("encoder.pth", "depth.pth"):
        shutil.copy(origen / f, PESOS_SCARED / f)

### ¿Este video sirve para evaluar?

Cada modelo guardó en su `historial.json` qué videos usó para entrenar y validar. Un video de esa lista, o de su
mismo **segmento** (por ejemplo `c2_cecum_t1_v2` si se entrenó con `c2_cecum_t1_v1`, porque recorren el mismo
tejido), daría métricas demasiado buenas. Se puede ver, pero no cuenta como evaluación.

In [ ]:
segmento = lambda v: re.sub(r"_v\d+.*$", "", v)
usados = set()
for m, c in MODELOS_E.items():
    div = json.load(open(c / "historial.json"))["division"]
    usados |= set(div["entrenamiento"]) | set(div["validacion"])
choca = [u for u in sorted(usados) if u == VIDEO or segmento(u) == segmento(VIDEO)]
EVALUACION_VALIDA = not choca
if choca:
    print(f"AVISO: {VIDEO} comparte segmento con videos usados para entrenar o validar: {choca}.\n"
          "Sirve para verlo, pero sus métricas NO son una evaluación válida.")
else:
    print(f"{VIDEO}: ningún modelo lo vio al entrenar ni validar. Evaluación válida.")

## 1. Herramientas

Son **las mismas funciones** del notebook de entrenamiento del Avance 5, para que la inferencia sea idéntica:
- **Lectura y máscaras:** el frame se lleva a 320×256, y las máscaras (El Meslouhi, Arnold) se calculan sobre el frame
  a 288×288, como al reentrenar Endo-STTN.
- **AF-SfMLearner:** carga, predicción de profundidad en mm y métricas con *median scaling*.
- **Endo-STTN:** la inferencia oficial (frames vecinos + frames de referencia), por bloques.

In [ ]:
import cv2
import numpy as np
import pandas as pd

ANCHO, ALTO, LADO = 320, 256, 288
PROF_MAX_MM = 100.0


def indice(nombre):
    m = re.search(r"(\d+)", Path(nombre).stem)
    return int(m.group(1)) if m else None


def leer_rgb(zf, ruta):
    return cv2.cvtColor(cv2.imdecode(np.frombuffer(zf.read(ruta), np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def leer_profundidad(zf, ruta):
    """Profundidad en mm (float32). 0 = sin dato (incluye píxeles saturados)."""
    d = cv2.imdecode(np.frombuffer(zf.read(ruta), np.uint8), cv2.IMREAD_UNCHANGED)
    d = d[..., 0] if d.ndim == 3 else d
    if d.dtype == np.uint16:
        mm = d.astype(np.float32) * (PROF_MAX_MM / 65535.0)
        mm[d == 65535] = 0
        return mm
    return d.astype(np.float32)


def _srgb_a_xyz(img):
    lin = np.where(img <= 0.04045, img / 12.92, ((img + 0.055) / 1.055) ** 2.4)
    m = np.array([[0.4124564, 0.3575761, 0.1804375],
                  [0.2126729, 0.7151522, 0.0721750],
                  [0.0193339, 0.1191920, 0.9503041]])
    return lin @ m.T


def _dilatar_1px(m):
    return cv2.dilate(m.astype(np.uint8), np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.uint8)).astype(bool)


def mascara_meslouhi(rgb):
    """El Meslouhi et al. (2011): realce por min/max, CIE-XYZ, reflejo donde Y >= Y/(X+Y+Z)."""
    img = rgb.astype(np.float64) / 255
    mn, mx = img.min(-1), img.max(-1)
    s = np.divide(mn, mx, out=np.zeros_like(mn), where=mx > 0)
    xyz = _srgb_a_xyz(s[..., None] * img)
    suma = xyz.sum(-1)
    y = np.divide(xyz[..., 1], suma, out=np.full_like(suma, np.inf), where=suma > 0)
    return _dilatar_1px(xyz[..., 1] >= y)


def mascara_arnold(rgb, t_abs=240, t_cand=200, t_rel=1.2, mediana=31):
    """Arnold et al. (2010), versión simplificada: detector absoluto + detector relativo."""
    img = rgb.astype(np.float32)
    gris = img @ np.array([0.2989, 0.5870, 0.1140], np.float32)
    p95 = lambda c: max(float(np.percentile(c, 95)), 1.0)
    r_g, r_b = p95(img[..., 1]) / p95(gris), p95(img[..., 2]) / p95(gris)
    sobre = lambda t: (img[..., 1] > r_g * t) | (img[..., 2] > r_b * t) | (gris > t)
    candidatos = cv2.dilate(sobre(t_cand).astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
    sin_reflejos = cv2.medianBlur(cv2.inpaint(rgb, candidatos, 5, cv2.INPAINT_TELEA), mediana).astype(np.float32)
    relativo = ((img / (sin_reflejos + 1)).max(-1) > t_rel) & (gris > t_cand * 0.75)
    return _dilatar_1px(sobre(t_abs) | relativo)


METODOS = {"meslouhi": mascara_meslouhi, "arnold": mascara_arnold}


def mascara(rgb_320, metodo):
    """Máscara a 320x256 calculada sobre el frame a 288x288, como al reentrenar Endo-STTN."""
    m = METODOS[metodo](cv2.resize(rgb_320, (LADO, LADO), interpolation=cv2.INTER_AREA))
    return cv2.resize(m.astype(np.uint8), (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST).astype(bool)

In [ ]:
import warnings
import torch
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")   # aviso de 'pretrained' en AF-SfMLearner
import networks                                  # AF-SfMLearner
from model.sttn import InpaintGenerator          # Endo-STTN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROF_MIN, PROF_MAX = 0.1, PROF_MAX_MM
NEIGHBOR_STRIDE, REF_LENGTH = 5, 10   # como en Endo-STTN/test.py


def cargar_estimador(carpeta):
    enc = networks.ResnetEncoder(18, False)
    sd = torch.load(Path(carpeta) / "encoder.pth", map_location=DEVICE)
    enc.load_state_dict({k: v for k, v in sd.items() if k in enc.state_dict()})
    dec = networks.DepthDecoder(num_ch_enc=enc.num_ch_enc, scales=range(4))
    dec.load_state_dict(torch.load(Path(carpeta) / "depth.pth", map_location=DEVICE))
    return enc.to(DEVICE).eval(), dec.to(DEVICE).eval()


@torch.no_grad()
def predecir_video(enc, dec, rgb):
    """(N,H,W,3) uint8 -> profundidad en mm (N,H,W) float32. Igual que disp_to_depth de AF-SfMLearner."""
    salida = []
    for i in range(0, len(rgb), 16):
        x = torch.from_numpy(np.array(rgb[i:i + 16])).permute(0, 3, 1, 2).float().div(255).to(DEVICE)
        disp = dec(enc(x))[("disp", 0)]
        salida.append((1 / (1 / PROF_MAX + (1 / PROF_MIN - 1 / PROF_MAX) * disp))[:, 0].cpu().numpy())
    return np.concatenate(salida)


def metricas(pred, gt, region=None):
    """AbsRel y a1 con median scaling. La escala se calcula con toda la imagen y luego se mide en la región."""
    valido = (gt > 0) & (gt <= PROF_MAX)
    if valido.sum() < 10:
        return None
    escala = np.median(gt[valido]) / np.median(pred[valido])
    if region is not None:
        valido &= region
        if valido.sum() < 10:
            return None
    p, g = np.clip(pred[valido].astype(np.float64) * escala, PROF_MIN, PROF_MAX), gt[valido].astype(np.float64)
    return {"abs_rel": float(np.mean(np.abs(p - g) / g)), "a1": float(np.mean(np.maximum(p / g, g / p) < 1.25))}


def cargar_endo(pesos):
    sd = torch.load(pesos, map_location=DEVICE)
    sd = {k.removeprefix("module."): v for k, v in sd.get("netG", sd).items()}
    modelo = InpaintGenerator().to(DEVICE)
    modelo.load_state_dict(sd)
    return modelo.eval()


@torch.no_grad()
def limpiar_bloque(modelo, rgb, mask):
    """rgb (T,H,W,3) uint8, mask (T,H,W) bool -> (T,H,W,3) uint8 con los reflejos rellenados."""
    T, H, W = mask.shape
    f = np.stack([cv2.resize(x, (LADO, LADO), interpolation=cv2.INTER_AREA) for x in rgb])
    m = np.stack([cv2.resize(x.astype(np.uint8), (LADO, LADO), interpolation=cv2.INTER_NEAREST) for x in mask])
    feats = torch.from_numpy(f).permute(0, 3, 1, 2).float().div(255).mul(2).sub(1).to(DEVICE)
    masks = torch.from_numpy(m).float().unsqueeze(1).to(DEVICE)
    enc = modelo.encoder(feats * (1 - masks))
    _, c, fh, fw = enc.shape
    enc = enc.view(1, T, c, fh, fw)
    acum, cuenta = np.zeros((T, LADO, LADO, 3), np.float32), np.zeros(T, np.float32)
    for k in range(0, T, NEIGHBOR_STRIDE):
        vec = list(range(max(0, k - NEIGHBOR_STRIDE), min(T, k + NEIGHBOR_STRIDE + 1)))
        ids = vec + [i for i in range(0, T, REF_LENGTH) if i not in vec]
        pred = torch.tanh(modelo.decoder(modelo.infer(enc[0, ids], masks[ids])[:len(vec)]))
        pred = ((pred + 1) / 2).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy() * 255
        for i, idx in enumerate(vec):
            acum[idx] += pred[i]
            cuenta[idx] += 1
    pred = acum / cuenta[:, None, None, None]
    out = rgb.copy()
    for t in range(T):
        relleno = cv2.resize(pred[t], (W, H), interpolation=cv2.INTER_LINEAR)
        out[t][mask[t]] = np.clip(relleno[mask[t]], 0, 255).astype(np.uint8)
    return out


def limpiar_video(modelo, rgb, mask):
    return np.concatenate([limpiar_bloque(modelo, rgb[i:i + BLOQUE], mask[i:i + BLOQUE])
                           for i in range(0, len(rgb), BLOQUE)])


print("Dispositivo:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu (sin GPU: será muy lento)")

## 2. Leer el video

Se leen los frames de `rgb/` **en orden** (y `depth/` si existe), a 320×256. El orden importa: Endo-STTN usa los
frames vecinos para rellenar.

In [ ]:
ruta_zip = CARPETA_RAW / f"{VIDEO}.zip"
assert ruta_zip.exists(), f"No encuentro {ruta_zip}"
with zipfile.ZipFile(ruta_zip) as zf:
    nombres = zf.namelist()
    rgb_rutas = {indice(n): n for n in nombres if n.startswith("rgb/") and n.lower().endswith((".png", ".jpg"))}
    dep_rutas = {indice(n): n for n in nombres if n.startswith("depth/") and n.lower().endswith((".tiff", ".tif", ".png"))}
    indices = sorted(set(rgb_rutas) - {None})[::PASO][:MAX_FRAMES or None]
    TIENE_PROFUNDIDAD = bool(dep_rutas) and all(i in dep_rutas for i in indices)
    t0 = time.time()
    RGB = np.stack([cv2.resize(leer_rgb(zf, rgb_rutas[i]), (ANCHO, ALTO), interpolation=cv2.INTER_AREA) for i in indices])
    GT = (np.stack([cv2.resize(leer_profundidad(zf, dep_rutas[i]), (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST)
                    for i in indices]) if TIENE_PROFUNDIDAD else None)
print(f"{VIDEO}: {len(RGB)} frames (paso {PASO}) en {time.time() - t0:.0f} s | "
      f"profundidad real: {'sí' if TIENE_PROFUNDIDAD else 'no (solo profundidad relativa, sin métricas)'}")

## 3. Aplicar los modelos

Para cada modelo E, los dos pasos del pipeline, en este orden:
1. **Inpainting:** máscara con **su** método y limpieza con **su** `sttn.pth`.
2. **Profundidad:** su `encoder.pth` y `depth.pth` sobre los frames limpios.

La referencia **A** (si `INCLUIR_A`) es el estimador original sin ajustar, sobre los frames sin limpiar: muestra el
punto de partida.

In [ ]:
RES = {}     # nombre -> {"mask", "limpio", "prof"}
for m, carpeta in MODELOS_E.items():
    t0 = time.time()
    mask = np.stack([mascara(x, m) for x in RGB])
    endo = cargar_endo(carpeta / "sttn.pth")
    limpio = limpiar_video(endo, RGB, mask)
    del endo; torch.cuda.empty_cache()
    enc, dec = cargar_estimador(carpeta)
    RES[f"E · {m}"] = {"mask": mask, "limpio": limpio, "prof": predecir_video(enc, dec, limpio)}
    del enc, dec; torch.cuda.empty_cache()
    print(f"E · {m}: reflejos {100 * mask.mean():.2f} % de los píxeles | {time.time() - t0:.0f} s")
if INCLUIR_A:
    enc, dec = cargar_estimador(PESOS_SCARED)
    RES["A · original"] = {"mask": None, "limpio": RGB, "prof": predecir_video(enc, dec, RGB)}
    del enc, dec; torch.cuda.empty_cache()
NOMBRES = list(RES)
UNION = np.logical_or.reduce([r["mask"] for r in RES.values() if r["mask"] is not None])   # reflejos de cualquier máscara

## 4. Métricas (solo si el video trae `depth/`)

Por frame, con *median scaling* por frame (igual que en la evaluación del Avance 5): AbsRel y δ<1.25 en la imagen
completa y en los **reflejos** (unión de las dos máscaras, la misma región para todos los modelos).

Además, el **parpadeo**: cuánto cambia la profundidad relativa entre frames consecutivos (media de |log dₜ − log dₜ₋₁|).
El estimador procesa cada frame por separado, así que puede "temblar" más que la profundidad real; por eso se compara
contra el parpadeo de la propia profundidad real.

In [ ]:
from IPython.display import Markdown, display


def parpadeo(prof, validos=None):
    """Media de |log d_t - log d_{t-1}| (cada frame normalizado por su mediana: solo cuenta la forma)."""
    vals = []
    for t in range(1, len(prof)):
        a, b = prof[t - 1].astype(np.float64), prof[t].astype(np.float64)
        v = (a > 0) & (b > 0) if validos is None else validos[t - 1] & validos[t]
        if v.sum() > 10:
            la, lb = np.log(a[v] / np.median(a[v])), np.log(b[v] / np.median(b[v]))
            vals.append(np.mean(np.abs(la - lb)))
    return float(np.mean(vals)) if vals else float("nan")


if TIENE_PROFUNDIDAD:
    filas = []
    for nombre, r in RES.items():
        for t in range(len(RGB)):
            for region, reg in (("imagen", None), ("reflejos", UNION[t])):
                if (x := metricas(r["prof"][t], GT[t], reg)):
                    filas.append({"modelo": nombre, "frame": indices[t], "region": region, **x})
    POR_FRAME = pd.DataFrame(filas)
    POR_FRAME.to_csv(SALIDA / "metricas_por_frame.csv", index=False)
    resumen = POR_FRAME.pivot_table(index="modelo", columns="region", values=["abs_rel", "a1"]).round(4)
    resumen.columns = [f"{m} · {r}" for m, r in resumen.columns]
    valido_gt = GT > 0
    resumen["parpadeo"] = [round(parpadeo(RES[n]["prof"], valido_gt), 4) for n in resumen.index]
    resumen.loc["profundidad real", "parpadeo"] = round(parpadeo(GT, valido_gt), 4)
    aviso = "" if EVALUACION_VALIDA else "\n\n**AVISO:** el video comparte segmento con el entrenamiento; no es una evaluación válida."
    texto = f"### {VIDEO} · {len(RGB)} frames (paso {PASO}){aviso}\n\n" + resumen.to_markdown()
    open(SALIDA / "resumen.md", "w").write(texto + "\n")
    display(Markdown(texto))
else:
    print("Sin profundidad real: no hay métricas. Parpadeo (sin referencia):",
          {n: round(parpadeo(r["prof"]), 4) for n, r in RES.items()})

In [ ]:
import matplotlib.pyplot as plt

if TIENE_PROFUNDIDAD:
    COLOR = {"E · meslouhi": "#eb6834", "E · arnold": "#1baf7a", "A · original": "#6250d6"}
    fig, ax = plt.subplots(1, 2, figsize=(17, 4), sharex=True)
    for a, region in zip(ax, ["imagen", "reflejos"]):
        for nombre in NOMBRES:
            d = POR_FRAME[(POR_FRAME.modelo == nombre) & (POR_FRAME.region == region)]
            a.plot(d.frame, d.abs_rel, lw=1.5, color=COLOR.get(nombre), label=nombre)
        a.set_title(f"AbsRel por frame · {region}"); a.set_xlabel("frame"); a.grid(alpha=0.25); a.legend()
    plt.tight_layout(); plt.savefig(SALIDA / "absrel_por_frame.png", dpi=90); plt.show()

## 5. Visor frame por frame

Mueve el control para recorrer el video. Por cada modelo: el frame limpio (contorno cian = máscara de ese modelo), su
profundidad y, si hay profundidad real, el error relativo. Los colores usan **un mismo rango para todo el video**, así
que los cambios de color entre frames son cambios reales de profundidad.

Para la profundidad se usa **una sola escala por video y por modelo** (mediana de la real / mediana de la predicha en
todos los frames). Sin profundidad real, se muestra la profundidad relativa.

In [ ]:
from matplotlib import colormaps

# Escala única por modelo y rango de colores fijo para todo el video
ESCALA = {}
for nombre, r in RES.items():
    if TIENE_PROFUNDIDAD:
        v = (GT > 0) & (GT <= PROF_MAX)
        ESCALA[nombre] = float(np.median(GT[v]) / np.median(r["prof"][v]))
    else:
        ESCALA[nombre] = float(1 / np.median(r["prof"]))
todas = np.concatenate([(r["prof"][::max(1, len(RGB) // 50)] * ESCALA[n]).ravel() for n, r in RES.items()])
if TIENE_PROFUNDIDAD:
    todas = np.concatenate([todas, GT[::max(1, len(RGB) // 50)][GT[::max(1, len(RGB) // 50)] > 0]])
VMIN, VMAX = np.percentile(todas, [2, 98])
UNIDAD = "mm" if TIENE_PROFUNDIDAD else "relativa"


def colorear(x, vmin, vmax, mapa):
    """Arreglo 2D -> imagen RGB uint8 con un rango fijo (NaN en negro)."""
    norm = np.clip((x - vmin) / (vmax - vmin), 0, 1)
    img = (colormaps[mapa](np.nan_to_num(norm))[..., :3] * 255).astype(np.uint8)
    img[np.isnan(x)] = 0
    return img


def paneles(t):
    """Lista de filas; cada fila es una lista de (imagen RGB, título)."""
    fila0 = [(RGB[t], f"original · frame {indices[t]}")]
    if TIENE_PROFUNDIDAD:
        fila0.append((colorear(np.where(GT[t] > 0, GT[t], np.nan), VMIN, VMAX, "magma_r"), "profundidad real"))
    filas = [fila0]
    for nombre, r in RES.items():
        p = r["prof"][t] * ESCALA[nombre]
        fila = [(r["limpio"][t], f"{nombre}: limpio" if r["mask"] is not None else f"{nombre}: sin limpiar"),
                (colorear(p, VMIN, VMAX, "magma_r"), f"{nombre}: profundidad")]
        if TIENE_PROFUNDIDAD:
            err = np.where(GT[t] > 0, np.abs(p - GT[t]) / np.maximum(GT[t], 1e-3), np.nan)
            fila.append((colorear(err, 0, 0.5, "inferno"), f"{nombre}: error relativo (media {np.nanmean(err):.3f})"))
        filas.append(fila)
    return filas


def mostrar(t):
    filas = paneles(t)
    columnas = max(len(f) for f in filas)
    fig, ax = plt.subplots(len(filas), columnas, figsize=(4 * columnas, 3.3 * len(filas)), squeeze=False)
    for i, fila in enumerate(filas):
        for j in range(columnas):
            ax[i, j].axis("off")
            if j < len(fila):
                ax[i, j].imshow(fila[j][0]); ax[i, j].set_title(fila[j][1], fontsize=9)
        nombre = NOMBRES[i - 1] if i else None
        if nombre and RES[nombre]["mask"] is not None:
            ax[i, 0].contour(RES[nombre]["mask"][t], [0.5], colors="cyan", linewidths=0.5)
    plt.suptitle(f"{VIDEO} · profundidad de {VMIN:.1f} a {VMAX:.1f} ({UNIDAD}); error de 0 a 50 %", y=1.0)
    plt.tight_layout(); plt.show()


from ipywidgets import interact, IntSlider
interact(mostrar, t=IntSlider(value=int(np.argmax(UNION.reshape(len(UNION), -1).mean(1))), min=0, max=len(RGB) - 1,
                              step=1, continuous_update=False, description="frame"));

## 6. Video `.mp4`

Se arma el video completo con el mismo diseño del visor (una fila por modelo) y se guarda en Drive. Con `PASO = 1`
dura lo mismo que el original a `FPS = 30`.

In [ ]:
def lienzo(t, alto=ALTO, ancho=ANCHO, banda=22):
    """Un cuadro del video: las filas del visor pegadas, con el título de cada panel arriba."""
    filas = paneles(t)
    columnas = max(len(f) for f in filas)
    out = np.full((len(filas) * (alto + banda), columnas * ancho, 3), 255, np.uint8)
    for i, fila in enumerate(filas):
        for j, (img, titulo) in enumerate(fila):
            y, x = i * (alto + banda), j * ancho
            out[y + banda:y + banda + alto, x:x + ancho] = img
            cv2.putText(out, titulo[:44], (x + 4, y + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (11, 11, 11), 1, cv2.LINE_AA)
        nombre = NOMBRES[i - 1] if i else None
        if nombre and RES[nombre]["mask"] is not None:
            y = i * (alto + banda) + banda
            contornos, _ = cv2.findContours(RES[nombre]["mask"][t].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            panel = np.ascontiguousarray(out[y:y + alto, 0:ancho])
            cv2.drawContours(panel, contornos, -1, (0, 255, 255), 1)
            out[y:y + alto, 0:ancho] = panel
    return out


primero = lienzo(0)
H, W = primero.shape[:2]
temporal, destino = LOCAL / f"{VIDEO}_tmp.mp4", SALIDA / f"{VIDEO}_end_to_end.mp4"
escritor = cv2.VideoWriter(str(temporal), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W + W % 2, H + H % 2))
t0 = time.time()
for t in range(len(RGB)):
    cuadro = lienzo(t) if t else primero
    cuadro = cv2.copyMakeBorder(cuadro, 0, H % 2, 0, W % 2, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    escritor.write(cv2.cvtColor(cuadro, cv2.COLOR_RGB2BGR))
escritor.release()
# H.264 para que se reproduzca en el navegador y en presentaciones
r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(temporal), "-vcodec", "libx264",
                    "-pix_fmt", "yuv420p", "-crf", "23", str(destino)])
if r.returncode != 0:
    shutil.copy(temporal, destino)
print(f"Video: {destino} ({len(RGB)} cuadros, {len(RGB) / FPS:.1f} s, {time.time() - t0:.0f} s en armarlo)")

In [ ]:
from IPython.display import Video
if destino.stat().st_size < 60e6:
    display(Video(str(destino), embed=True, width=min(W, 1100)))
else:
    print("El video es grande para mostrarlo aquí: ábrelo desde Drive:", destino)

## 7. Cómo leerlo

- **El relleno** debería verse estable de un frame a otro: Endo-STTN usa los frames vecinos.
- **La profundidad** puede "temblar" un poco: el estimador procesa cada frame por separado. Compara el **parpadeo**
  de cada modelo con el de la profundidad real; si es mucho mayor, es una limitación del estimador, no del video.
- **Dentro de los reflejos** (contorno cian) es donde E debería notarse frente a A: sin limpiar, los reflejos suelen
  verse como zonas "muy cerca".
- **E · meslouhi vs E · arnold:** compara las columnas de error y el AbsRel en reflejos. Recuerda que cada uno usa su
  propia máscara; la tabla mide los dos en la misma región (la unión).
- Si apareció el **AVISO** de segmento compartido, el video sirve para verlo, pero sus números no cuentan como
  evaluación.